In [0]:
%pip install --upgrade typing_extensions>=4.9.0
%pip install mlflow databricks-vectorsearch "langgraph>=0.2.0" "langchain-openai>=0.2.0" openai
dbutils.library.restartPython()

In [0]:
import mlflow
import sys
import os
sys.path.append('/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent/')

from gdpr_agent import config, build_agent

In [0]:
class GDPRAgentWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from gdpr_agent import config, build_agent
        
        openai_key = os.environ.get("OPENAI_API_KEY")
        config.setup(openai_key)
        self.app = build_agent()
    
    def predict(self, context, model_input):
        import pandas as pd
        
        if isinstance(model_input, pd.DataFrame):
            question = model_input["question"].iloc[0]
        else:
            question = model_input.get("question", "")
        
        state = {
            "original_question": question,
            "current_query": question,
            "retrieved_context": "",
            "generated_answer": "",
            "retrieval_loop_count": 0,      
            "generation_loop_count": 0    
        }
        
        final = self.app.invoke(state)
        return pd.DataFrame([{
            "answer": final["generated_answer"],
            "context": final["retrieved_context"]
        }])

print("✅ Wrapper class defined")

In [0]:
os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(scope="openai", key="GDPR_agent")

mlflow.set_experiment("/Users/vernonc.lam@gmail.com/gdpr-agent-model")

# Create example input/output for signature
import pandas as pd

example_input = pd.DataFrame({
    "question": ["What is GDPR Article 5?"]
})

example_output = pd.DataFrame({
    "answer": ["Example answer about GDPR Article 5"],
    "context": ["Example context"]
})

# Infer signature from examples
signature = mlflow.models.infer_signature(example_input, example_output)

# Declare Vector Search dependencies
from mlflow.models.resources import DatabricksVectorSearchIndex

resources = [
    DatabricksVectorSearchIndex(
        index_name="main.default.gdpr_law_vector_index"
    ),
    DatabricksVectorSearchIndex(
        index_name="main.default.gdpr_fines_vector_index"
    ),
    DatabricksVectorSearchIndex(
        index_name="main.default.privacy_policy_vector_index"
    )
]

with mlflow.start_run(run_name="gdpr-agent-v3"):
    # Package the gdpr_agent code WITH the model
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=GDPRAgentWrapper(),
        signature=signature,
        code_paths=["/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent/gdpr_agent"],
        resources=resources,  # ← ADD THIS!
        pip_requirements=[
            "typing_extensions>=4.9.0",
            "databricks-vectorsearch",
            "langgraph>=0.2.0",
            "langchain-openai>=0.2.0",
            "openai>=1.0.0"
        ],
        registered_model_name="gdpr_agent_model"
    )

print("✅ Model logged with Vector Search resources! Now redeploy via Serving UI.")

In [0]:
# Test evaluation harness manually
import json
import sys
sys.path.insert(0, '/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent/eval_harness')

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Load first 3 test cases
with open("/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent/evaluation_data/golden_set_v3_production_30.json", 'r') as f:
    dataset = json.load(f)

# Test on 3 cases
for test_case in dataset["test_cases"][:3]:
    print(f"\n{'='*80}")
    print(f"Case: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    
    # Query endpoint
    response = w.serving_endpoints.query(
        name="gdpr-agent-endpoint",
        dataframe_records=[{"question": test_case["question"]}]
    )
    
    print(f"\nAgent Answer: {response.predictions[0]['answer'][:200]}...")
    print(f"Expected: {test_case['expected_behavior']}")

In [0]:
# Evaluation scoring function
def evaluate_case(test_case: dict, agent_response: dict) -> dict:
    """
    Score agent response against expected behavior
    """
    expected = test_case.get("expected_behavior", {})
    answer = agent_response.get("answer", "")
    context = agent_response.get("context", "")
    
    scores = {
        "source_correct": 0,
        "content_match": 0,
        "total": 0
    }
    feedback = []
    
    # Check 1: Did it retrieve from expected sources?
    expected_sources = expected.get("sources", [])
    source_found = any(src.lower() in context.lower() for src in expected_sources)
    
    if source_found:
        scores["source_correct"] = 1
        feedback.append(f"✅ Retrieved from expected sources: {expected_sources}")
    else:
        feedback.append(f"❌ Expected sources {expected_sources} not found in context")
    
    # Check 2: Does answer contain expected articles/keywords?
    must_include = expected.get("must_retrieve_from_articles", []) + expected.get("must_include_in_answer", [])
    
    if must_include:
        found = [item for item in must_include if item.lower() in (answer + context).lower()]
        coverage = len(found) / len(must_include) if must_include else 0
        scores["content_match"] = coverage
        
        if coverage >= 0.7:
            feedback.append(f"✅ Found {len(found)}/{len(must_include)} expected items")
        else:
            missing = set(must_include) - set(found)
            feedback.append(f"❌ Missing: {missing}")
    else:
        scores["content_match"] = 1  # No requirements
    
    # Overall score
    scores["total"] = (scores["source_correct"] + scores["content_match"]) / 2
    
    return {
        "scores": scores,
        "feedback": feedback,
        "passed": scores["total"] >= 0.7
    }

print("✅ Evaluation function defined")

In [0]:
# Test with scoring - ALL 30 CASES
results = []

for test_case in dataset["test_cases"]: 
    print(f"\n{'='*80}")
    print(f"Case: {test_case['id']}")
    print(f"Question: {test_case['question'][:80]}...")
    
    # Query endpoint
    response = w.serving_endpoints.query(
        name="gdpr-agent-endpoint",
        dataframe_records=[{"question": test_case["question"]}]
    )
    
    agent_response = {
        "answer": response.predictions[0]['answer'],
        "context": response.predictions[0]['context']
    }
    
    # Evaluate
    eval_result = evaluate_case(test_case, agent_response)
    
    # Record
    results.append({
        "case_id": test_case["id"],
        "category": test_case["category"],
        "passed": eval_result["passed"],
        "score": eval_result["scores"]["total"],
        "feedback": " | ".join(eval_result["feedback"])
    })
    
    status = "✅ PASS" if eval_result["passed"] else "❌ FAIL"
    print(f"{status} | Score: {eval_result['scores']['total']:.2f}")

# Summary
import pandas as pd
results_df = pd.DataFrame(results)
print(f"\n{'='*80}")
print(f"📊 BASELINE EVALUATION RESULTS")
print(f"{'='*80}")
print(f"Total Cases: {len(results_df)}")
print(f"Passed: {results_df['passed'].sum()}")
print(f"Failed: {(~results_df['passed']).sum()}")
print(f"Pass Rate: {results_df['passed'].mean()*100:.1f}%")
print(f"Avg Score: {results_df['score'].mean():.2f}")

print(f"\n📈 By Category:")
print(results_df.groupby('category').agg({
    'passed': ['sum', 'count'],
    'score': 'mean'
}).round(2))

# Save baseline results
results_df.to_csv("/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent/evaluation_data/baseline_results.csv", index=False)
print(f"\n💾 Results saved to baseline_results.csv")

display(results_df)

In [0]:
# Debug cell - check fines retrieval scores
from gdpr_agent import config
from gdpr_agent.tools import tool_search_historical_fines

config.setup(dbutils.secrets.get(scope="openai", key="GDPR_agent"))

test_query = "What companies have been fined for GDPR violations?"

results = tool_search_historical_fines(query_text=test_query, top_k=5)
fine_rows = results.get('result', {}).get('data_array', [])

print("Fines search results:")
for i, row in enumerate(fine_rows):
    score = row[-1]
    text = row[1][:200] if len(row) > 1 else "N/A"
    print(f"\n[{i+1}] Score: {score:.3f}")
    print(f"Content preview: {text}...")
    print(f"Would be included? {score > 0.35}")

In [0]:
# Check what's actually in the fines index
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# Query the underlying table
display(spark.sql("""
    SELECT source_file_name, translated_text 
    FROM main.default.gdpr_translated_chunks 
    WHERE translated_text LIKE '%Google%' 
       OR translated_text LIKE '%Meta%'
       OR translated_text LIKE '%British Airways%'
    LIMIT 10
"""))

In [0]:
from eval_harness import EvaluationRunner
from eval_harness.utils import (
    generate_report, 
    print_category_breakdown, 
    filter_failed_cases
)

runner = EvaluationRunner(
    endpoint_name="gdpr-agent-endpoint",
    dataset_path="/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent/evaluation_data/golden_set_v3_production_30.json"
)

results_df = runner.run_evaluation()

# Use utilities
print_category_breakdown(results_df)
report = generate_report(results_df, output_path="./eval_report.txt")
failed = filter_failed_cases(results_df)

display(failed)